# 11 · Screen mid-vs-late — EHR conditioning ở block 5,7 vs 10,11

08 chỉ ra: block 5–7 còn head cục bộ + còn depth phía sau; block 10–11 thì bão hòa, attention
gần global, gần hết depth. 07 đã tiêm EHR ở block 10–11 → null. Câu **chưa** thử: block 5–7.

Đây là **screen tầng 1** (tái dùng H5 + split hiện có, ~1 ngày), không phải confirmatory.
Chỉ dương mới xây sealed confirmatory 5-seed.

## Bốn arm — control capacity ở MỖI vị trí

| arm | adapter ở block (1-idx) | context |
|---|---|---|
| `M_U` | 5, 7 | learned constant (EHR = NULL) |
| `M_E` | 5, 7 | patient EHR (full) |
| `L_U` | 10, 11 | learned constant |
| `L_E` | 10, 11 | patient EHR (full) |

Cả 4 cùng số tham số (2 CondAdapter + 1 EhrEncoder + 1 decoder-copy), cùng init, cùng H5 warm-start.

## Cổng KÉP (sửa lỗi cổng-I-đơn)

`I_location` một mình sai: nếu mid & late cùng giúp đều thì `I≈0` dù conditioning có ích. Nên:

```
G_mid = M_E − M_U        G_late = L_E − L_U        I_location = G_mid − G_late
```

Cổng chạy confirmatory = **G_mid ≥ +0.002** (có đáng không), `I_location` **chọn claim**
(mid tốt hơn / conditioning có ích nói chung). Điều kiện cần: `real EHR > shuffle`.

## Quyết định (khóa trước run)

| kết quả | hành động |
|---|---|
| `G_mid ≥ +0.002` và `I ≥ +0.002`, real>shuffle | Case A — sealed confirmatory, claim "mid > late" |
| `G_mid ≥ +0.002`, `G_late ≥ +0.002`, `I < +0.002` | Case B — confirmatory, claim "conditioning có ích" |
| `G_mid < +0.002`, real≈shuffle | Case C — **DỪNG** |
| bất ổn, dấu lệch giữa seed | Case D — thêm 2 seed |

**Phạm vi:** **Full EHR** (labs + micro org/spec + vitals + como) — micro (`org_name`) là
feature chỉ-đích chẩn đoán, phù hợp nhất cho cơ chế "context steer attention"; loại nó theo
"micro≈0 ở late-fusion 06" là mâu thuẫn vì 11 tồn tại chính vì deep≠late. Screen trên H5 cố
định + split hiện có ⇒ null "đủ để dừng kỹ thuật"; dương vẫn cần confirmatory.

**Decoder: DELAYED UNFREEZE** — freeze warmup ngắn → unfreeze với LR thấp (0.1× adapter) +
L2-SP về H5. Cân bằng false-negative (frozen thuần) và phá-H5 (unfreeze-từ-đầu, đã thấy ở 07).
EHR encoder **luôn train**. Estimand `G_mid`, `I_location` vẫn hợp lệ vì mọi arm cùng schedule.


## C0 · Config + preregistration

**Hạ tầng:** viết cho **RTX PRO 6000 Blackwell (96 GB)**. Nếu chạy trên máy không phải Colab,
chỉnh `DATA_DIR` / `LOCAL_IMG` / `SHARD_DIR` về đường dẫn local; nếu ảnh đã giải nén sẵn ở
`LOCAL_IMG` thì C0-bis là no-op.

In [ ]:
import os, json, math, time, pickle, random, shutil, tarfile, contextlib
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'                                   # Blackwell/local: dat DATA_DIR o day
OUT_DIR   = os.path.join(DATA_DIR, 'cohort_v2')
LOCAL_IMG = os.environ.get('LOCAL_IMG', '/content/cxr_local')
SHARD_DIR = os.path.join(DATA_DIR, 'cxr_560_shards')
BB_CKPT   = os.path.join(DATA_DIR, 'lpft_v2', 'raddino_mld_db_ft_best.pt')            # 04b backbone
H5_CKPT   = os.path.join(DATA_DIR, 'token_fusion', 'checkpoints', 'H5_s42_best.pt')   # H5 head (06b), giong 07
SC_DIR    = os.path.join(DATA_DIR, 'midvslate'); os.makedirs(SC_DIR, exist_ok=True)
os.makedirs(f'{SC_DIR}/checkpoints', exist_ok=True); os.makedirs(f'{SC_DIR}/metrics', exist_ok=True)

W_F, W_L = 0.7, 0.3
SEEDS    = [42, 1337, 2024]

# --- decoder: DELAYED UNFREEZE (freeze warmup -> unfreeze LR thap + L2-SP ve H5) ---
FREEZE_EHR_ENCODER    = False        # encoder LUON train
DECODER_WARMUP_EPOCHS = 1            # freeze decoder trong warmup, roi unfreeze
LR_ADAPTER            = 1e-4         # = LR_ENCODER
LR_DECODER            = 1e-5         # 0.1x adapter
L2SP_DECODER          = 1e-5         # neo decoder ve H5
CLIP_DECODER, CLIP_OTHER = 1.0, 5.0

# --- vi tri adapter: 1-INDEXED trong config/report, doi sang 0-indexed mot lan ---
MID_1IDX, LATE_1IDX = (5, 7), (10, 11)
MID  = tuple(b-1 for b in MID_1IDX)                  # 0-indexed
LATE = tuple(b-1 for b in LATE_1IDX)

PREREG = dict(
    primary_gate='G_mid = M_E - M_U (val study-level macro mAP)',
    location='I_location = (M_E-M_U) - (L_E-L_U)',
    require='real EHR > shuffle EHR',
    mid_blocks_1idx=list(MID_1IDX), late_blocks_1idx=list(LATE_1IDX),
    threshold_practical=0.002, mpre=0.002,
    d_e=128, bottleneck=64, epochs=8, patience=2,
    lr_adapter=LR_ADAPTER, lr_decoder=LR_DECODER, l2sp=L2SP_DECODER,
    decoder_warmup_epochs=DECODER_WARMUP_EPOCHS, wd=1e-2, warmup_frac=0.05, loss='ASL',
    effective_batch=256, features='full-EHR (labs+micro+vitals+como)', decoder='delayed_unfreeze',
    seeds=SEEDS, note='screen tang 1: fixed H5+04b, existing split, val-only gate')
json.dump(PREREG, open(f'{SC_DIR}/preregistration.json','w'), indent=1)

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev, '|', torch.cuda.get_device_name(0) if dev=='cuda' else 'CPU')
print(f'MID blocks (1-idx) {MID_1IDX} -> 0-idx {MID} | LATE {LATE_1IDX} -> {LATE}')
print('PREREG khoa ->', f'{SC_DIR}/preregistration.json')

Mounted at /content/drive
device: cuda | NVIDIA RTX PRO 6000 Blackwell Server Edition
MID blocks (1-idx) (5, 7) -> 0-idx (4, 6) | LATE (10, 11) -> (9, 10)
PREREG khoa -> /content/drive/MyDrive/MIMIC_CXR/planB_out/midvslate/preregistration.json


In [ ]:
CLEAN_OLD = False     # <- dat True MOT LAN de don labs-only cu, roi tra ve False
if CLEAN_OLD:
    import glob
    _old = f'{SC_DIR}/metrics/screen.json'
    if os.path.exists(_old):
        os.rename(_old, f'{SC_DIR}/metrics/screen_LABSONLY.json'); print('doi ten screen.json -> screen_LABSONLY.json')
    for f in glob.glob(f'{SC_DIR}/checkpoints/_resume_s*.pt'):
        os.remove(f); print('xoa', f)
    print('sach -> chay full-EHR tu dau')
else:
    print('CLEAN_OLD=False (giu nguyen). Dat True 1 lan neu con checkpoint labs-only cu.')

CLEAN_OLD=False (giu nguyen). Dat True 1 lan neu con checkpoint labs-only cu.


## C0-bis · Stage ảnh (Colab). Trên Blackwell local: no-op nếu ảnh đã ở `LOCAL_IMG`.

In [ ]:
os.makedirs(LOCAL_IMG, exist_ok=True)
_have = len([f for f in os.listdir(LOCAL_IMG) if f.endswith('.jpg')])
if _have > 1000 and not os.path.isdir(SHARD_DIR):
    print(f'{_have:,} anh da co san o {LOCAL_IMG}; khong co SHARD_DIR -> bo qua staging')
else:
    tars = sorted(f for f in os.listdir(SHARD_DIR) if f.endswith('.tar')) if os.path.isdir(SHARD_DIR) else []
    assert tars or _have > 1000, f'khong co shard o {SHARD_DIR} va {LOCAL_IMG} rong -> can data'
    todo = [f for f in tars if not os.path.exists(os.path.join(LOCAL_IMG, f'.done_{f}'))]
    print(f'shards: {len(tars)} | to stage: {len(todo)}')
    for f in tqdm(todo, desc='staging shards', unit='shard'):
        tmp = f'/content/{f}'
        shutil.copyfile(os.path.join(SHARD_DIR, f), tmp)
        with tarfile.open(tmp) as t: t.extractall(LOCAL_IMG)
        os.remove(tmp); open(os.path.join(LOCAL_IMG, f'.done_{f}'), 'w').close()
    print('local images:', len([f for f in os.listdir(LOCAL_IMG) if f.endswith('.jpg')]))

shards: 347 | to stage: 347


staging shards:   0%|          | 0/347 [00:00<?, ?shard/s]

/tmp/ipykernel_6582/2868312186.py:13: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  with tarfile.open(tmp) as t: t.extractall(LOCAL_IMG)


local images: 346578


## C1 · Cohort + **Full EHR** data-prep (verbatim 06b·C1-C2) + tier per-tier

Lift nguyên máy móc EHR của 06b (đã chạy đúng): labs, micro org/spec (ragged EmbeddingBag),
vitals, comorbidity, recency, tiers T1/T2/T3. Cột lấy từ `ehr` + `ehr_vocab.pkl`; C1 assert
từng phần, thiếu cột sẽ báo rõ tại đây.

In [ ]:
meta   = json.load(open(os.path.join(OUT_DIR, 'cohort_meta.json')))
LABELS = meta['labels']; assert len(LABELS) == 13
ehr = pd.read_parquet(os.path.join(OUT_DIR, 'cohort_study_level.parquet')).sort_values('study_id').reset_index(drop=True)
N = len(ehr)
Y_STUDY  = ehr[LABELS].to_numpy('float32')
EHR_ROW  = {int(s): i for i, s in enumerate(ehr.study_id.to_numpy())}

# ---- vocab + full-EHR arrays (verbatim 06b·C2) ----
vocab = pickle.load(open(os.path.join(OUT_DIR, 'ehr_vocab.pkl'), 'rb'))
ORG_V, SPEC_V = vocab['org_vocab_size'], vocab['spec_vocab_size']
MISSING = vocab['reserved']['MISSING']

def as_id_lists(col):
    out = []
    for v in ehr[col].to_numpy():
        a = np.asarray(v, np.int64).ravel() if v is not None and np.ndim(v) else np.array([], np.int64)
        out.append(a if a.size else np.array([MISSING], np.int64))
    return out
RAGGED = {'t1_org':('t1_org_ids',ORG_V), 't1_spec':('t1_spec_ids',SPEC_V),
          't2_spec':('t2_spec_ids',SPEC_V), 't3_org':('t3_org_ids',ORG_V)}
bags = {k: as_id_lists(c) for k,(c,_) in RAGGED.items()}

def codes(series, cats):
    m = {c.lower(): i for i, c in enumerate(cats)}
    return series.astype(str).str.lower().map(lambda x: m.get(x, 0)).to_numpy('int64'), len(cats)
set_id,  _ = codes(ehr['setting'],        ['inpatient','ED','outpatient'])
rec_id,  _ = codes(ehr['t3_recency_bin'], ['none','7-30d','30-90d'])
stat_id    = ehr['t1_status'].to_numpy('int64')

BIN = ['t2_pending','t2_time_imprecise','t1_same_encounter']
binv = {b: ehr[b].to_numpy('float32') for b in BIN}
tr_mask = (ehr.split == 'train').to_numpy()
def zscore_train(x, use_log=True):
    z = np.log1p(np.clip(x.astype('float32'), 0, None)) if use_log else x.astype('float32')
    mu, sd = z[tr_mask].mean(), z[tr_mask].std() or 1.0
    return ((z - mu)/sd).astype('float32')
COUNTS = ['t2_n_pending','t3_n_specimens','t4_n_prior_positive_ever',
          't4_years_since_first_culture','t4_n_prior_encounters']
cnts = {c: zscore_train(ehr[c].to_numpy()) for c in COUNTS}

_z = lambda k: np.zeros((N, k), 'float32')
LABS   = vocab['labs']
lab_val  = np.stack([ehr[f'{c}_value'].to_numpy('float32') for c in LABS], 1)
lab_mask = np.stack([ehr[f'{c}_mask'].to_numpy('float32')  for c in LABS], 1)
VITALS = [c for c in vocab.get('vitals', []) if f'{c}_value' in ehr.columns]
vit_val  = np.stack([ehr[f'{c}_value'].to_numpy('float32') for c in VITALS], 1) if VITALS else _z(0)
vit_mask = np.stack([ehr[f'{c}_mask'].to_numpy('float32')  for c in VITALS], 1) if VITALS else _z(0)
COMO_COLS = [c for c in (vocab.get('comorbidity', []) + ['como_mask']) if c in ehr.columns]
como_arr  = np.stack([ehr[c].to_numpy('float32') for c in COMO_COLS], 1) if COMO_COLS else _z(0)
REC1 = ['t1_days_since_last_specimen','t1_days_since_last_positive']; REC3 = ['t3_days_since_last_positive']
rec_block = lambda ns: (np.stack([ehr[f'{c}_value'].to_numpy('float32') for c in ns], 1),
                        np.stack([ehr[f'{c}_mask'].to_numpy('float32')  for c in ns], 1))
rec1_v, rec1_m = rec_block(REC1); rec3_v, rec3_m = rec_block(REC3)
MICRO_TIERS = (1, 2, 3)

cf = pickle.load(open(os.path.join(OUT_DIR, 'class_freq.pkl'), 'rb'))
TRAIN_PREV = np.asarray(cf['class_freq'])/cf['num_train']
TIER = np.where(TRAIN_PREV<0.02,'tail', np.where(TRAIN_PREV<0.10,'medium','head'))
print(f'cohort {N:,} | labs {len(LABS)} | vitals {len(VITALS)} | como {COMO_COLS} | '
      f'micro org/spec {ORG_V}/{SPEC_V} | tiers {MICRO_TIERS}')
print(f'tiers nhan: {dict(zip(*np.unique(TIER,return_counts=True)))}')

vm = pd.read_parquet(os.path.join(OUT_DIR, 'view_map.parquet'))
vm = vm[vm.view_group.isin(['frontal','lateral'])]
_dup = [c for c in ('subject_id','split','setting') if c in vm.columns]
man = vm.drop(columns=_dup).merge(ehr[['study_id','subject_id','split']], on='study_id', how='inner')
before = len(man)
man = man[man.dicom_id.map(lambda d: os.path.exists(f'{LOCAL_IMG}/{d}.jpg'))].sort_values('dicom_id').reset_index(drop=True)
assert len(man), f'ALL {before:,} images missing -> chay C0-bis'
assert (before-len(man))/before < 0.01, f'{before-len(man)} images missing -> chay C0-bis'
man['ehr_row'] = man.study_id.map(EHR_ROW).astype('int64')
IDX = {s: np.where(man.split.to_numpy()==s)[0] for s in ('train','val','test')}
print('images by split:', {s: int(len(IDX[s])) for s in IDX})
print('NOTE: test index dung nhung KHONG BAO GIO dung trong screen nay.')

cohort 211,130 | labs 7 | vitals 6 | como ['como_diabetes', 'como_hiv', 'como_mask'] | micro org/spec 311/82 | tiers (1, 2, 3)
tiers nhan: {np.str_('head'): np.int64(7), np.str_('medium'): np.int64(5), np.str_('tail'): np.int64(1)}
images by split: {'train': 242497, 'val': 51880, 'test': 52201}
NOTE: test index dung nhung KHONG BAO GIO dung trong screen nay.


## C2 · Dataset đầy đủ (verbatim 06b·C4) — ragged bag `(cat,offsets)`, `perm` shuffle

`perm` tráo **mọi** feature qua `e`; `y` LUÔN từ `ehr_row` thật (nhãn không đi theo shuffle).
`to_dev` giữ tuple để cặp `(cat, offsets)` của EmbeddingBag lên device đúng.

In [ ]:
from transformers import AutoImageProcessor
from torchvision import transforms as T
from PIL import Image
proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MODEL_SIZE = proc.crop_size['height']; MEAN, STD = proc.image_mean[0], proc.image_std[0]
train_tf = T.Compose([T.RandomResizedCrop(MODEL_SIZE, scale=(0.85,1.0), ratio=(0.95,1.05),
                                          interpolation=T.InterpolationMode.BICUBIC),
                      T.RandomRotation(7), T.ToTensor(), T.Normalize([MEAN]*3,[STD]*3)])
eval_tf  = T.Compose([T.CenterCrop(MODEL_SIZE), T.ToTensor(), T.Normalize([MEAN]*3,[STD]*3)])

class ImgDS(Dataset):
    def __init__(self, idx, tf, perm=None): self.idx, self.tf, self.perm = np.asarray(idx), tf, perm
    def __len__(self): return len(self.idx)
    def __getitem__(self, k):
        r = man.iloc[self.idx[k]]
        im = Image.open(f'{LOCAL_IMG}/{r.dicom_id}.jpg').convert('L')
        im = Image.merge('RGB', (im, im, im))
        tr = int(r.ehr_row)                                # y LUON tu row that
        e  = tr if self.perm is None else int(self.perm[tr])
        d = dict(image=self.tf(im), y=torch.from_numpy(Y_STUDY[tr]),
                 set_id=set_id[e], rec_id=rec_id[e], stat_id=stat_id[e],
                 lab_val=torch.from_numpy(lab_val[e]), lab_mask=torch.from_numpy(lab_mask[e]),
                 vit_val=torch.from_numpy(vit_val[e]), vit_mask=torch.from_numpy(vit_mask[e]),
                 como=torch.from_numpy(como_arr[e]),
                 rec1_v=torch.from_numpy(rec1_v[e]), rec1_m=torch.from_numpy(rec1_m[e]),
                 rec3_v=torch.from_numpy(rec3_v[e]), rec3_m=torch.from_numpy(rec3_m[e]))
        for _bn in BIN:    d[_bn] = binv[_bn][e]
        for _c in COUNTS:  d[_c]  = cnts[_c][e]
        for nm in RAGGED:  d[nm]  = torch.from_numpy(bags[nm][e])
        return d

def collate(bb):
    o = dict(image=torch.stack([x['image'] for x in bb]), y=torch.stack([x['y'] for x in bb]),
             set_id=torch.tensor([x['set_id'] for x in bb]),
             rec_id=torch.tensor([x['rec_id'] for x in bb]),
             stat_id=torch.tensor([x['stat_id'] for x in bb]))
    for kk in ('lab_val','lab_mask','vit_val','vit_mask','como','rec1_v','rec1_m','rec3_v','rec3_m'):
        o[kk] = torch.stack([x[kk] for x in bb])
    for kk in BIN + COUNTS: o[kk] = torch.tensor([x[kk] for x in bb], dtype=torch.float32)
    for nm in RAGGED:
        lens = [len(x[nm]) for x in bb]
        o[nm] = (torch.cat([x[nm] for x in bb]) if sum(lens) else torch.empty(0, dtype=torch.int64),
                 torch.tensor([0]+lens[:-1]).cumsum(0))
    return o

def loader(split, tf, shuffle, bs, perm=None, workers=8):
    return DataLoader(ImgDS(IDX[split], tf, perm), batch_size=bs, shuffle=shuffle, collate_fn=collate,
                      num_workers=workers, pin_memory=(dev=='cuda'), persistent_workers=(workers>0))
to_dev = lambda b: {k:(tuple(t.to(dev, non_blocking=True) for t in v) if isinstance(v, tuple)
                       else v.to(dev, non_blocking=True)) for k, v in b.items()}
def shuffle_perm(seed=0):
    rng = np.random.default_rng(seed); p = np.arange(N)
    for s in ('train','val','test'):                       # tráo TRONG từng split
        rows = np.unique(man.iloc[IDX[s]].ehr_row.to_numpy()); pr = rows.copy(); rng.shuffle(pr); p[rows] = pr
    return p
print('loaders ready (full-EHR dataset)')

preprocessor_config.json:   0%|          | 0.00/756 [00:00<?, ?B/s]

loaders ready (full-EHR dataset)


## C3 · Backbone đông cứng + `forward_shared` / `forward_tail` (hai vị trí)

`SHARED_UPTO = min(mọi block adapter) = 4` (0-idx) ⇒ blocks 0–3 chạy chung `no_grad`. Trong
`forward_tail`, block trước adapter đầu tiên chạy `no_grad` (không tham số học nào phụ thuộc),
từ adapter đầu tiên trở đi mới bật grad ⇒ arm late (adapter 9,10) không tốn grad cho block 4–8.

In [ ]:
from transformers import AutoModel
backbone = AutoModel.from_pretrained('microsoft/rad-dino').to(dev).eval()
assert os.path.exists(BB_CKPT), f'04b backbone khong thay: {BB_CKPT}'
_sd = torch.load(BB_CKPT, map_location=dev, weights_only=False)['model']
_bb = {k[9:]: v for k,v in _sd.items() if k.startswith('backbone.')}
assert _bb, 'no backbone.* keys in 04b checkpoint'
backbone.load_state_dict(_bb, strict=True)
for p in backbone.parameters(): p.requires_grad = False
LAYERS = backbone.encoder.layer; n_blocks = len(LAYERS); D = backbone.config.hidden_size
assert D == 768 and max(LATE) < n_blocks
SHARED_UPTO = min(MID + LATE)
print(f'blocks {n_blocks} | shared no_grad 0..{SHARED_UPTO-1} | adapters MID {MID} LATE {LATE}')

def _layer(h, i):
    o = LAYERS[i](h); return o[0] if isinstance(o,(tuple,list)) else o

def forward_shared(x):
    with torch.no_grad():
        h = backbone.embeddings(x)
        for i in range(SHARED_UPTO): h = _layer(h, i)
    return h.detach()

def forward_tail(h, adapter_blocks=(), adapters=None, e=None):
    first = min(adapter_blocks) if adapter_blocks else n_blocks
    for i in range(SHARED_UPTO, n_blocks):
        with (torch.no_grad() if i < first else contextlib.nullcontext()):
            h = _layer(h, i)
        if adapters is not None and str(i) in adapters:
            h = adapters[str(i)](h, e)
    return backbone.layernorm(h)

# G0: manual loop == stock forward
_x = torch.randn(2, 3, MODEL_SIZE, MODEL_SIZE, device=dev)
with torch.no_grad():
    ref = backbone(pixel_values=_x).last_hidden_state
    got = forward_tail(forward_shared(_x))
md_ = (ref-got).abs().max().item()
assert torch.allclose(ref, got, atol=1e-4), f'manual != stock (max {md_:.2e})'
print(f'G0 manual forward == backbone(x).last_hidden_state  max|diff| {md_:.2e}  OK')

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

blocks 12 | shared no_grad 0..3 | adapters MID (4, 6) LATE (9, 10)
G0 manual forward == backbone(x).last_hidden_state  max|diff| 0.00e+00  OK


## C4 · Full EhrEncoder (verbatim 06b) + CondAdapter (verbatim 07)

`EhrEncoder`: labs + vitals + como + micro (ragged EmbeddingBag org/spec) + recency + tiers
T1/T2/T3 → `e[128]`. `CondAdapter`: `W_up` zero-init ⇒ `ΔZ=0` ⇒ identity tuyệt đối ở step 0.

In [ ]:
Act = nn.GELU

class EhrEncoder(nn.Module):
    def __init__(self, d_e=PREREG['d_e'], d_hidden=256, dropout=0.2):
        super().__init__(); T = set(MICRO_TIERS); self.T = T
        self.mods = nn.ModuleDict(); d = 0
        def bag(key, vsize, dim):
            self.mods[key] = nn.EmbeddingBag(vsize, dim, mode='sum'); return dim
        self.set_emb = nn.Embedding(3, 4); d += 4
        d += 2*len(LABS) + 2*len(VITALS) + len(COMO_COLS)
        self.n_vit, self.n_como = len(VITALS), len(COMO_COLS)
        if 1 in T:
            d += bag('t1_org', ORG_V, 16) + bag('t1_spec', SPEC_V, 8)
            self.stat_emb = nn.Embedding(4, 4); d += 4 + 1 + 2*len(REC1)
        if 2 in T: d += bag('t2_spec', SPEC_V, 8) + 3
        if 3 in T:
            d += bag('t3_org', ORG_V, 16); self.rec_emb = nn.Embedding(3, 4)
            d += 4 + 2*len(REC3) + 1
        self.in_dim = d
        self.mlp = nn.Sequential(nn.Linear(d, d_hidden), nn.LayerNorm(d_hidden), Act(),
                                 nn.Dropout(dropout),
                                 nn.Linear(d_hidden, d_e), nn.LayerNorm(d_e), Act())
    def _bag(self, k, b):
        flat, off = b[k]; m = self.mods[k]
        return m(flat, off)
    def forward(self, b):
        T = self.T
        p = [self.set_emb(b['set_id']), b['lab_val'], b['lab_mask']]
        if self.n_vit:  p += [b['vit_val'], b['vit_mask']]
        if self.n_como: p += [b['como']]
        if 1 in T: p += [self._bag('t1_org',b), self._bag('t1_spec',b), self.stat_emb(b['stat_id']),
                         b['t1_same_encounter'][:,None], b['rec1_v'], b['rec1_m']]
        if 2 in T: p += [self._bag('t2_spec',b), b['t2_pending'][:,None],
                         b['t2_time_imprecise'][:,None], b['t2_n_pending'][:,None]]
        if 3 in T: p += [self._bag('t3_org',b), self.rec_emb(b['rec_id']), b['rec3_v'], b['rec3_m'],
                         b['t3_n_specimens'][:,None]]
        return self.mlp(torch.cat(p, 1))

class CondAdapter(nn.Module):
    def __init__(self, d=768, r=PREREG['bottleneck'], d_e=PREREG['d_e']):
        super().__init__()
        self.ln=nn.LayerNorm(d); self.down=nn.Linear(d,r); self.gen=nn.Linear(d_e,2*r); self.up=nn.Linear(r,d)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
        nn.init.zeros_(self.gen.weight); nn.init.zeros_(self.gen.bias)
    def forward(self, Z, e):
        u = self.down(self.ln(Z)); g,b = self.gen(e).chunk(2,-1)
        u = (1+g[:,None,:])*u + b[:,None,:]
        return Z + self.up(F.gelu(u))
print('EhrEncoder in_dim:', EhrEncoder().in_dim, '-> d_e', PREREG['d_e'],
      '| params:', sum(p.numel() for p in EhrEncoder().parameters()))
print('adapter params/block:', sum(p.numel() for p in CondAdapter().parameters()))

EhrEncoder in_dim: 100 -> d_e 128 | params: 70824
adapter params/block: 117184


## C5 · ArmModel × 4 + D0 — warm-start H5, decoder DELAYED-UNFREEZE + NULL_BATCH

`M_U/M_E/L_U/L_E` cùng init adapter+encoder (adapter theo vị trí), decoder cả 4 == H5 (cùng
warm-start). Decoder **trainable** (freeze/unfreeze theo epoch ở C9) + neo L2-SP về H5.
`uncond` nạp `NULL_BATCH` (mọi feature missing, hằng qua sample) qua **cùng** encoder ⇒
learned-constant control param-matched.

In [ ]:
import sys, types
ML_SRC = '/content/ml_decoder.py'
if not os.path.exists(ML_SRC):
    !wget -q -O {ML_SRC} https://raw.githubusercontent.com/Alibaba-MIIL/ML_Decoder/main/src_files/ml_decoder/ml_decoder.py
sys.path.insert(0, '/content')
import ml_decoder as _mld
from ml_decoder import MLDecoder
if not hasattr(_mld.TransformerDecoderLayerOptimal, '_patched'):
    _oi=_mld.TransformerDecoderLayerOptimal.__init__
    def _init(self,*a,**kw): _oi(self,*a,**kw); self.self_attn=types.SimpleNamespace(batch_first=False)
    _mld.TransformerDecoderLayerOptimal.__init__=_init
    _of=_mld.TransformerDecoderLayerOptimal.forward
    def _fwd(self,tgt,memory,**kw): kw.pop('tgt_is_causal',None); kw.pop('memory_is_causal',None); return _of(self,tgt,memory,**kw)
    _mld.TransformerDecoderLayerOptimal.forward=_fwd
    _mld.TransformerDecoderLayerOptimal._patched=True

assert os.path.exists(H5_CKPT), f'H5 head khong thay: {H5_CKPT} (token_fusion/checkpoints/H5_s42_best.pt tu 06b)'
H5_HEAD = torch.load(H5_CKPT, map_location=dev, weights_only=False)['model']
def _sub(prefix): return {k[len(prefix):]: v for k,v in H5_HEAD.items() if k.startswith(prefix)}
assert _sub('norm.') and _sub('mld.'), f'H5 checkpoint thieu norm.*/mld.* keys: {list(H5_HEAD)[:5]}'
print('H5 head <- token_fusion/checkpoints/H5_s42_best.pt  | norm', len(_sub('norm.')), 'mld', len(_sub('mld.')))

def NULL_BATCH(B, device):
    """Batch 'moi thu missing' — hang qua moi sample -> unconditional control."""
    z1  = lambda w: torch.zeros(B, w, device=device)
    sc  = lambda: torch.zeros(B, device=device)
    idl = lambda: torch.zeros(B, dtype=torch.long, device=device)
    d = dict(set_id=idl(), stat_id=idl(), rec_id=idl(),
             lab_val=z1(lab_val.shape[1]), lab_mask=z1(lab_mask.shape[1]),
             vit_val=z1(vit_val.shape[1]), vit_mask=z1(vit_mask.shape[1]),
             como=z1(como_arr.shape[1]),
             rec1_v=z1(rec1_v.shape[1]), rec1_m=z1(rec1_m.shape[1]),
             rec3_v=z1(rec3_v.shape[1]), rec3_m=z1(rec3_m.shape[1]))
    for kk in ('t1_same_encounter','t2_pending','t2_time_imprecise','t2_n_pending','t3_n_specimens'):
        d[kk] = sc()
    for nm in RAGGED:                                       # 1 token MISSING/sample -> emb[MISSING] hang
        d[nm] = (torch.full((B,), MISSING, dtype=torch.long, device=device),
                 torch.arange(B, dtype=torch.long, device=device))
    return d

class ArmModel(nn.Module):
    def __init__(self, mode, blocks):                      # mode: plain|uncond|cond
        super().__init__(); self.mode=mode; self.blocks=tuple(blocks)
        self.norm = nn.LayerNorm(768); self.mld = MLDecoder(num_classes=len(LABELS),
                     initial_num_features=768, decoder_embedding=768, num_of_groups=len(LABELS))
        self.norm.load_state_dict(_sub('norm.')); self.mld.load_state_dict(_sub('mld.'))   # decoder trainable
        if mode != 'plain':
            self.enc = EhrEncoder()
            self.adapters = nn.ModuleDict({str(i): CondAdapter() for i in self.blocks})
    def forward(self, z_shared, batch):
        if self.mode=='plain':
            h = forward_tail(z_shared)
        else:
            e = self.enc(batch) if self.mode=='cond' else self.enc(NULL_BATCH(z_shared.shape[0], z_shared.device))
            h = forward_tail(z_shared, self.blocks, self.adapters, e)
        return self.mld(self.norm(h[:, 1:]))

def _copy_pos(dst_md, src_md):
    # copy THEO VI TRI (M_U key {4,6}, L_* key {9,10}) — load_state_dict theo key se sai.
    kd = sorted(dst_md, key=int); ks = sorted(src_md, key=int); assert len(kd)==len(ks)
    for a, b in zip(kd, ks): dst_md[a].load_state_dict(src_md[b].state_dict())

def build_arms(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    mu = ArmModel('uncond', MID).to(dev)
    A = {'M_U': mu, 'M_E': ArmModel('cond', MID).to(dev),
         'L_U': ArmModel('uncond', LATE).to(dev), 'L_E': ArmModel('cond', LATE).to(dev)}
    for t in ('M_E','L_U','L_E'):                          # cung init adapter (vi tri) + encoder; decoder == H5 auto
        _copy_pos(A[t].adapters, mu.adapters); A[t].enc.load_state_dict(mu.enc.state_dict())
    return A

D0 = ArmModel('plain', ()).to(dev).eval()
for p in D0.parameters(): p.requires_grad_(False)          # anchor: H5 nguyen ban, khong train
dec_params = lambda m: list(m.norm.parameters()) + list(m.mld.parameters())   # thu tu on dinh
H5_DEC = [p.detach().clone() for p in dec_params(D0)]       # neo L2-SP (== H5)

_A0 = build_arms(42)
n = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)
print('trainable params (adapter+enc+decoder):', {t: f'{n(m)/1e6:.3f}M' for t,m in _A0.items()})
assert len({n(m) for m in _A0.values()}) == 1, 'ca 4 arm phai cung so tham so hoc'
_m = _A0['M_E']
print(f'  adapter {sum(p.numel() for p in _m.adapters.parameters())} '
      f'| enc {sum(p.numel() for p in _m.enc.parameters())} '
      f'| decoder {sum(p.numel() for p in dec_params(_m))}')
print('4 arms param-matched  OK  | decoder DELAYED-UNFREEZE (trainable, warm-start H5, L2-SP neo)')

H5 head <- token_fusion/checkpoints/H5_s42_best.pt  | norm 2 mld 19
trainable params (adapter+enc+decoder): {'M_U': '6.423M', 'M_E': '6.423M', 'L_U': '6.423M', 'L_E': '6.423M'}
  adapter 234368 | enc 70824 | decoder 6127629
4 arms param-matched  OK  | decoder DELAYED-UNFREEZE (trainable, warm-start H5, L2-SP neo)


## C6 · Cổng — identity step-0, EHR đọc labs+micro, NULL_BATCH hằng, L2-SP, ASL

In [ ]:
_b = next(iter(loader('val', eval_tf, False, 4, workers=0))); _b = to_dev(_b)
_z = forward_shared(_b['image'])
for m in _A0.values(): m.eval()
D0.eval()
with torch.no_grad():
    y0 = D0(_z, _b)
    for t, m in _A0.items():
        d = (m(_z, _b) - y0).abs().max().item()
        print(f'  {t}: |logit - D0| step-0 = {d:.2e}', 'OK' if d < 1e-4 else '!! KHONG identity')
        assert d < 1e-4
    e1 = _A0['M_E'].enc(_b)
    # EHR doc LABS
    _bl = dict(_b); _bl['lab_val'] = _b['lab_val'] + 3.0
    d_lab = (e1 - _A0['M_E'].enc(_bl)).abs().mean().item()
    # EHR doc MICRO (ragged bag t1_org): doi org id -> e doi
    _cat, _off = _b['t1_org']; _bm = dict(_b)
    _bm['t1_org'] = (torch.full_like(_cat, (int(_cat.max())+1) % ORG_V), _off)
    d_mic = (e1 - _A0['M_E'].enc(_bm)).abs().mean().item()
    print(f'  EHR nhay: labs |Δe| {d_lab:.3f} | micro t1_org |Δe| {d_mic:.3f}  (ca hai > 0)')
    assert d_lab > 1e-3 and d_mic > 1e-4
    # NULL_BATCH hang qua sample + khac real
    e_null = _A0['M_E'].enc(NULL_BATCH(4, dev))
    print(f'  NULL_BATCH: std giua sample {e_null.std(0).max().item():.2e} (~0) | '
          f'|real-null| {(e1-e_null).abs().mean().item():.3f} (>0)')
    assert e_null.std(0).max().item() < 1e-4 and (e1-e_null).abs().mean().item() > 1e-3

# L2-SP = 0 khi decoder == H5, > 0 khi decoder troi
l2 = lambda m: sum(((p-ref)**2).sum() for p,ref in zip(dec_params(m), H5_DEC)).item()
l2_0 = l2(_A0['M_E'])
_p = next(iter(_A0['M_E'].norm.parameters())); _orig = _p.detach().clone()
with torch.no_grad(): _p.add_(0.01)
l2_1 = l2(_A0['M_E'])
with torch.no_grad(): _p.copy_(_orig)
print(f'  L2-SP: init {l2_0:.2e} (=0) | sau khi nhich decoder {l2_1:.2e} (>0)')
assert l2_0 < 1e-8 and l2_1 > 0

class ASLoss(nn.Module):
    def __init__(s, gn=4, gp=1, clip=0.05, eps=1e-8): super().__init__(); s.gn,s.gp,s.clip,s.eps=gn,gp,clip,eps
    def forward(s, x, y):
        y=y.float(); xp=torch.sigmoid(x); xn=1-xp
        if s.clip: xn=(xn+s.clip).clamp(max=1)
        los=y*torch.log(xp.clamp(min=s.eps))+(1-y)*torch.log(xn.clamp(min=s.eps))
        pt=xp*y+xn*(1-y); g=s.gp*y+s.gn*(1-y); return -(los*torch.pow(1-pt,g)).mean()
_a=ASLoss(); assert _a(torch.tensor([[-4.6]]),torch.tensor([[0.]])).item()<1e-3 and _a(torch.tensor([[4.6]]),torch.tensor([[0.]])).item()>1.0
print('GATES OK')

  M_U: |logit - D0| step-0 = 2.38e-07 OK
  M_E: |logit - D0| step-0 = 2.38e-07 OK
  L_U: |logit - D0| step-0 = 2.38e-07 OK
  L_E: |logit - D0| step-0 = 2.38e-07 OK
  EHR nhay: labs |Δe| 0.297 | micro t1_org |Δe| 0.191  (ca hai > 0)
  NULL_BATCH: std giua sample 0.00e+00 (~0) | |real-null| 0.311 (>0)
  L2-SP: init 0.00e+00 (=0) | sau khi nhich decoder 7.68e-02 (>0)
GATES OK


## C7 · Predict + pool 7:3 + per-tier AP

In [ ]:
from sklearn.metrics import average_precision_score
@torch.no_grad()
def predict(models, split, bs, perm=None):
    for m in models.values(): m.eval()
    P = {t: np.zeros((len(IDX[split]), len(LABELS)), 'float32') for t in models}
    pos = 0
    for b in tqdm(loader(split, eval_tf, False, bs, perm, workers=0), desc=f'pred[{split}]', leave=False):
        b = to_dev(b); z = forward_shared(b['image']); k = z.shape[0]
        for t, m in models.items(): P[t][pos:pos+k] = torch.sigmoid(m(z, b)).float().cpu().numpy()
        pos += k
    return P
def pool_study(p_img, split):
    sub = man.iloc[IDX[split]]; d = pd.DataFrame(p_img, columns=LABELS)
    d['study_id']=sub.study_id.to_numpy(); d['vg']=sub.view_group.to_numpy()
    sid = np.sort(d.study_id.unique())
    fr = d[d.vg=='frontal'].groupby('study_id')[LABELS].mean().reindex(sid)
    la = d[d.vg=='lateral'].groupby('study_id')[LABELS].mean().reindex(sid)
    hf = fr.notna().all(1).to_numpy().astype('float32'); hl = la.notna().all(1).to_numpy().astype('float32')
    den = (W_F*hf + W_L*hl)[:,None]
    return sid, (W_F*np.nan_to_num(fr.to_numpy('float32'))*hf[:,None]
                 + W_L*np.nan_to_num(la.to_numpy('float32'))*hl[:,None]) / den
def ap_tiers(p_img, split):
    sid, PS = pool_study(p_img, split)
    Yt = ehr.set_index('study_id').loc[sid, LABELS].to_numpy('float32')
    ap = np.array([average_precision_score(Yt[:,j], PS[:,j]) for j in range(len(LABELS))])
    return dict(macro=ap.mean(), head=ap[TIER=='head'].mean(),
                medium=ap[TIER=='medium'].mean(), tail=ap[TIER=='tail'].mean(), per=ap)
print('predict/pool/ap_tiers ready')

predict/pool/ap_tiers ready


## C8 · Train helpers + S0 smoke (2 pha: warmup decoder-frozen → unfreeze)

Helper dùng chung cho S0 và C9: `make_opt` (2 LR-group: adapter+enc 1e-4, decoder 1e-5),
`make_sched`, `set_decoder_trainable`, `l2sp_term`, `clip_arm` (clip riêng module). S0 xác
nhận: warmup grad decoder=0 / adapter+enc>0; sau unfreeze grad decoder>0; L2-SP hợp lệ.

In [ ]:
def make_opt(m):
    ad = list(m.adapters.parameters()) + list(m.enc.parameters())
    de = dec_params(m)
    def split(ps, lr):
        dec=[p for p in ps if p.ndim>1]; nod=[p for p in ps if p.ndim<=1]
        return [{'params':dec,'lr':lr,'weight_decay':PREREG['wd']},
                {'params':nod,'lr':lr,'weight_decay':0.0}]
    return torch.optim.AdamW(split(ad, LR_ADAPTER) + split(de, LR_DECODER))   # groups: ad_dec,ad_nod,de_dec,de_nod

def make_sched(opt, total, warm):
    cw = lambda s: s/max(warm,1) if s<warm else 0.5*(1+math.cos(math.pi*(s-warm)/max(total-warm,1)))
    const = lambda s: 1.0
    return torch.optim.lr_scheduler.LambdaLR(opt, [cw, cw, const, const])

def set_decoder_trainable(m, on):
    for p in dec_params(m): p.requires_grad_(on)

def l2sp_term(m):
    return sum(((p-ref)**2).sum() for p,ref in zip(dec_params(m), H5_DEC))

def clip_arm(m):
    dec = [p for p in dec_params(m) if p.grad is not None]
    oth = [p for p in (list(m.adapters.parameters())+list(m.enc.parameters())) if p.grad is not None]
    if dec: nn.utils.clip_grad_norm_(dec, CLIP_DECODER)
    if oth: nn.utils.clip_grad_norm_(oth, CLIP_OTHER)

S0 = False
if S0:
    A = build_arms(42); crit = ASLoss()
    dl = DataLoader(ImgDS(IDX['train'][:1500], train_tf), batch_size=16, shuffle=True,
                    collate_fn=collate, num_workers=4)
    for m in A.values(): set_decoder_trainable(m, False); m.train()        # WARMUP: decoder frozen
    opt = {t: make_opt(m) for t,m in A.items()}
    losses = {t: [] for t in A}; steps = 0; last = None
    for b in tqdm(dl, desc='S0-warmup', leave=False):
        b = to_dev(b); z = forward_shared(b['image']); last = (z, b)
        for t, m in A.items():
            opt[t].zero_grad(set_to_none=True)
            l = crit(m(z, b), b['y']); l.backward(); clip_arm(m); opt[t].step(); losses[t].append(l.item())
        steps += 1
        if steps >= 40: break
    g_ad  = sum(p.grad.abs().sum().item() for p in A['M_E'].adapters.parameters() if p.grad is not None)
    g_enc = sum(p.grad.abs().sum().item() for p in A['M_E'].enc.parameters()      if p.grad is not None)
    g_dec = sum(p.grad.abs().sum().item() for p in dec_params(A['M_E'])           if p.grad is not None)
    print(f'WARMUP grad: adapter {g_ad:.2e} (>0) | enc {g_enc:.2e} (>0) | decoder {g_dec:.2e} (=0 frozen)')
    assert g_ad > 0 and g_enc > 0 and g_dec == 0
    print('loss dau->cuoi:', {t: f'{np.mean(losses[t][:5]):.3f}->{np.mean(losses[t][-5:]):.3f}' for t in A})
    # UNFREEZE: decoder train + L2-SP -> decoder grad > 0
    for m in A.values(): set_decoder_trainable(m, True)
    z, b = last; m = A['M_E']; m.zero_grad(set_to_none=True)
    (crit(m(z, b), b['y']) + L2SP_DECODER*l2sp_term(m)).backward()
    g_dec2 = sum(p.grad.abs().sum().item() for p in dec_params(m) if p.grad is not None)
    print(f'UNFREEZE grad decoder {g_dec2:.2e} (>0) | L2-SP term {l2sp_term(m).item():.2e}')
    assert g_dec2 > 0
    print('S0 OK -> dat S0=False roi chay C9')

## C9 · Main screen — 3 seed × 4 arm, forward chung 0–3, delayed-unfreeze, resume

Mỗi arm optimizer(2 LR-group)/scheduler/early-stop RIÊNG. Đầu mỗi epoch set
`decoder.requires_grad = (ep ≥ WARMUP)` (resume-safe). L2-SP chỉ khi unfrozen. Clip riêng
module. Blackwell 96GB: `probe_batch` tự dò (đo ở trạng thái unfrozen = tốn bộ nhớ nhất).

In [ ]:
def probe_batch(cands=(16,32,48,64,96)):
    if dev!='cuda': return 8
    A=build_arms(42); crit=ASLoss(); ok=cands[0]
    for m in A.values(): set_decoder_trainable(m, True)                 # worst-case memory
    for B in cands:
        try:
            torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            bb=to_dev(collate([ImgDS(IDX['train'],train_tf)[k] for k in range(B)]))
            z=forward_shared(bb['image'])
            for m in A.values(): (crit(m(z,bb),bb['y'])+L2SP_DECODER*l2sp_term(m)).backward(); m.zero_grad(set_to_none=True)
            pk=torch.cuda.max_memory_allocated()/1e9; tot=torch.cuda.get_device_properties(0).total_memory/1e9
            print(f'  B={B:3d} peak {pk:5.1f}/{tot:.0f} GB ({pk/tot:5.1%})')
            ok = B if pk/tot < 0.80 else ok
            if pk/tot >= 0.80: break
        except torch.cuda.OutOfMemoryError:
            print(f'  B={B} OOM'); break
    del A; torch.cuda.empty_cache(); return ok

BATCH = probe_batch(); print('-> BATCH', BATCH)
EFF, EPOCHS, PATIENCE = PREREG['effective_batch'], PREREG['epochs'], PREREG['patience']

def ckpath(seed): return f'{SC_DIR}/checkpoints/_resume_s{seed}.pt'
def train_seed(seed, batch, epochs=EPOCHS):
    A = build_arms(seed); crit = ASLoss()
    dl = loader('train', train_tf, True, batch); accum = max(1, EFF//batch)
    total = epochs*math.ceil(len(dl)/accum); warm = int(PREREG['warmup_frac']*total)
    st = {}
    for t, m in A.items():
        opt = make_opt(m)
        st[t] = dict(m=m, opt=opt, sch=make_sched(opt, total, warm), best=-1., since=0, done=False, sd=None)
    start = 0
    if os.path.exists(ckpath(seed)):
        c = torch.load(ckpath(seed), map_location=dev, weights_only=False)
        if c.get('features') != PREREG['features']:
            raise RuntimeError(
                f"resume checkpoint la features='{c.get('features')}' nhung dang chay '{PREREG['features']}'.\n"
                f"    -> Xoa/doi ten {ckpath(seed)} roi chay lai.")
        if c.get('batch')!=batch: raise RuntimeError(f"resume batch {c.get('batch')} != {batch}; xoa {ckpath(seed)}")
        for t in st:
            st[t]['m'].load_state_dict(c[t]['model']); st[t]['opt'].load_state_dict(c[t]['opt'])
            st[t]['sch'].load_state_dict(c[t]['sch']); st[t].update(best=c[t]['best'], since=c[t]['since'], done=c[t]['done'], sd=c[t]['sd'])
        start = c['epoch']+1; print(f'RESUMED s{seed} @ep{start} ' + ' '.join(f'{t}:{st[t]["best"]:.4f}' for t in st))
        if start >= epochs or all(st[t]['done'] for t in st):     # seed da hoan tat
            for t in st: st[t]['m'].load_state_dict(st[t]['sd'])
            if 'final' in c:
                print(f'  seed {seed} hoan tat -> dung ket qua da luu (bo qua re-eval)'); return c['final']
            final = {t: ap_tiers(predict({t: st[t]['m']}, 'val', batch*2)[t], 'val') for t in st}
            c['final'] = final; torch.save(c, ckpath(seed)+'.tmp'); os.replace(ckpath(seed)+'.tmp', ckpath(seed))
            print(f'  seed {seed} hoan tat -> luu final, lan sau bo qua'); return final
    for ep in range(start, epochs):
        unfrozen = ep >= DECODER_WARMUP_EPOCHS
        tag = '[dec]' if unfrozen else '[warm]'
        for t in st:
            set_decoder_trainable(st[t]['m'], unfrozen)                 # resume-safe: tu epoch
            st[t]['m'].train()
        for i, b in enumerate(tqdm(dl, desc=f's{seed} ep{ep+1}/{epochs}{tag}', leave=False)):
            b = to_dev(b); z = forward_shared(b['image'])
            for t in st:
                e = st[t]
                if e['done']: continue
                loss = crit(e['m'](z, b), b['y'])
                if unfrozen: loss = loss + L2SP_DECODER*l2sp_term(e['m'])
                (loss/accum).backward()
                if (i+1)%accum==0 or i==len(dl)-1:
                    clip_arm(e['m']); e['opt'].step(); e['sch'].step(); e['opt'].zero_grad(set_to_none=True)
        Pv = predict({t: st[t]['m'] for t in st if not st[t]['done']}, 'val', batch*2)
        line = []
        for t in st:
            e = st[t]
            if e['done']: line.append(f'{t}:done'); continue
            mAP = ap_tiers(Pv[t], 'val')['macro']
            if mAP > e['best']: e['best'], e['since'], e['sd'] = mAP, 0, {k:v.detach().cpu().clone() for k,v in e['m'].state_dict().items()}
            else:
                e['since'] += 1
                if e['since'] >= PATIENCE: e['done'] = True
            line.append(f"{t}:{mAP:.4f}{'*' if e['since']==0 else ''}")
        torch.save({**{t: dict(model=st[t]['m'].state_dict(), opt=st[t]['opt'].state_dict(), sch=st[t]['sch'].state_dict(),
                              best=st[t]['best'], since=st[t]['since'], done=st[t]['done'], sd=st[t]['sd']) for t in st},
                    'epoch': ep, 'batch': batch, 'features': PREREG['features']}, ckpath(seed)+'.tmp'); os.replace(ckpath(seed)+'.tmp', ckpath(seed))
        print(f'  ep{ep+1}{tag} val ' + ' '.join(line) + '  [saved]')
        if all(st[t]['done'] for t in st): print('  all early-stopped'); break
    for t in st: st[t]['m'].load_state_dict(st[t]['sd'])
    final = {t: ap_tiers(predict({t: st[t]['m']}, 'val', batch*2)[t], 'val') for t in st}
    try:
        c = torch.load(ckpath(seed), map_location=dev, weights_only=False); c['final']=final
        torch.save(c, ckpath(seed)+'.tmp'); os.replace(ckpath(seed)+'.tmp', ckpath(seed))
    except Exception: pass
    return final

RESULT = {}
for s in SEEDS:
    print(f'\n===== seed {s} =====')
    RESULT[s] = {t: {k: float(v) for k,v in r.items() if k!='per'} for t,r in train_seed(s, BATCH).items()}
    json.dump(RESULT, open(f'{SC_DIR}/metrics/screen.json','w'), indent=1)
print('\nsaved -> metrics/screen.json')


  B= 16 peak   6.9/102 GB ( 6.8%)
  B= 32 peak  12.7/102 GB (12.5%)
  B= 48 peak  18.6/102 GB (18.2%)
  B= 64 peak  24.4/102 GB (23.9%)
  B= 96 peak  36.1/102 GB (35.4%)
-> BATCH 96

===== seed 42 =====
RESUMED s42 @ep6 M_U:0.4380 M_E:0.4416 L_U:0.4374 L_E:0.4406
  seed 42 hoan tat -> dung ket qua da luu (bo qua re-eval)

===== seed 1337 =====
RESUMED s1337 @ep3 M_U:0.4382 M_E:0.4413 L_U:0.4375 L_E:0.4410


s1337 ep4/8[dec]:   0%|          | 0/2527 [00:00<?, ?it/s]

pred[val]:   0%|          | 0/271 [00:00<?, ?it/s]

  ep4[dec] val M_U:0.4367 M_E:0.4409 L_U:0.4362 L_E:0.4404  [saved]
  all early-stopped


pred[val]:   0%|          | 0/271 [00:00<?, ?it/s]

...

pred[val]:   0%|          | 0/271 [00:00<?, ?it/s]

pred[val]:   0%|          | 0/271 [00:00<?, ?it/s]


===== seed 2024 =====


s2024 ep1/8[warm]:   0%|          | 0/2527 [00:00<?, ?it/s]

KeyboardInterrupt: 

## C10 · Kết quả — cổng kép G_mid + I_location, per-tier, decision matrix

In [ ]:
from scipy import stats as sstats
R = json.load(open(f'{SC_DIR}/metrics/screen.json'))
ss = sorted(R, key=lambda x:int(x))
def col(arm, tier='macro'): return np.array([R[s][arm][tier] for s in ss])
def ci(v): v=np.asarray(v,float); n=len(v); return (v.mean(), 0. if n<2 else sstats.t.ppf(0.975,n-1)*v.std(ddof=1)/np.sqrt(n))

print('===== val macro theo seed =====')
print(pd.DataFrame({t: col(t).round(4) for t in ('M_U','M_E','L_U','L_E')}, index=[f's{s}' for s in ss]).to_string())

print('\n===== per-tier (mean qua seed) =====')
print(pd.DataFrame({t: {ti: col(t,ti).mean().round(4) for ti in ('macro','head','medium','tail')}
                    for t in ('M_U','M_E','L_U','L_E')}).to_string())

# D0 (H5 anchor, seed-independent) -> tach hieu ung continued-training khoi EHR
d0 = ap_tiers(predict({'D0':D0}, 'val', 128)['D0'], 'val')['macro']
print(f'\nD0 (H5 anchor) val macro: {d0:.4f}')
print(f'  vs D0:  M_U {col("M_U").mean()-d0:+.4f} | M_E {col("M_E").mean()-d0:+.4f} | '
      f'L_U {col("L_U").mean()-d0:+.4f} | L_E {col("L_E").mean()-d0:+.4f}  (arm-D0 = adapter+continued-training)')

Gmid  = col('M_E')-col('M_U'); Glate = col('L_E')-col('L_U'); Iloc = Gmid-Glate
def show(nm, v):
    m,h = ci(v); print(f'  {nm:22s} {m:+.4f} ±{h:.4f}   [{", ".join(f"{x:+.4f}" for x in v)}]')
print('\n===== ESTIMANDS (khóa trước, ngưỡng +0.002) =====')
show('G_mid  = M_E-M_U', Gmid); show('G_late = L_E-L_U', Glate); show('I_loc  = G_mid-G_late', Iloc)
print('\n  per-tier G_mid:', {ti: round((col('M_E',ti)-col('M_U',ti)).mean(),4) for ti in ('head','medium','tail')})

TH = PREREG['threshold_practical']; gm, gl, il = Gmid.mean(), Glate.mean(), Iloc.mean()
consist = (np.sign(Gmid)==np.sign(gm)).sum()
print('\n===== DECISION =====')
if gm >= TH and il >= TH:
    print(f'  Case A: G_mid {gm:+.4f} ≥ {TH} và I {il:+.4f} ≥ {TH} -> sealed confirmatory, claim "mid > late"')
elif gm >= TH and gl >= TH:
    print(f'  Case B: mid & late cùng ≥ {TH}, I {il:+.4f} < {TH} -> confirmatory, claim "conditioning có ích"')
elif gm < TH:
    print(f'  Case C: G_mid {gm:+.4f} < {TH} -> DỪNG (mid không cứu được conditioning)')
else:
    print(f'  ambiguous -> xem real-vs-shuffle (C11) và Case D')
print(f'  dấu G_mid nhất quán {consist}/{len(ss)} seed')

## C11 · Diagnostics cơ chế — real vs shuffle, logit sensitivity, adapter activity

Điều kiện cần: `M_E real > M_E shuffle`. Vì decoder đã delayed-unfreeze (có cơ hội thích nghi),
không còn nhánh escalation. Đọc: `real≈shuffle` ⇒ capacity/shortcut, không phải patient-EHR;
`G_mid≈0` dù `M_E,M_U` cùng tăng ⇒ gain là continued-training (xem `arm-D0` ở C10), không quy EHR.

In [ ]:
c = torch.load(ckpath(SEEDS[0]), map_location=dev, weights_only=False)
A = build_arms(SEEDS[0])
for t in A: A[t].load_state_dict(c[t]['sd'])

perm = shuffle_perm(0)
Preal = predict({'M_E':A['M_E'],'L_E':A['L_E']}, 'val', BATCH*2)
Pshuf = predict({'M_E':A['M_E'],'L_E':A['L_E']}, 'val', BATCH*2, perm=perm)
for t in ('M_E','L_E'):
    r = ap_tiers(Preal[t],'val')['macro']; s = ap_tiers(Pshuf[t],'val')['macro']
    dlog = np.abs(Preal[t]-Pshuf[t]).mean()
    cc = np.corrcoef(Preal[t].ravel(), Pshuf[t].ravel())[0,1]
    print(f'  {t}: real {r:.4f} | shuffle {s:.4f} | real-shuffle {r-s:+.4f} | '
          f'mean|Δlogit| {dlog:.4f} corr {cc:.4f}')

# adapter activity: ||ΔZ||/||Z|| tai moi block adapter cua M_E
_b = to_dev(next(iter(loader('val', eval_tf, False, 8, workers=0))))
with torch.no_grad():
    z = forward_shared(_b['image']); e = A['M_E'].enc(_b); h = z
    print('\n  adapter activity (M_E):')
    for i in range(SHARED_UPTO, n_blocks):
        h_pre = _layer(h, i)
        if str(i) in A['M_E'].adapters:
            h_post = A['M_E'].adapters[str(i)](h_pre, e)
            ratio = (h_post-h_pre).norm(dim=-1).mean().item() / h_pre.norm(dim=-1).mean().item()
            g,bb = A['M_E'].adapters[str(i)].gen(e).chunk(2,-1)
            print(f'    block {i+1} (1-idx): ||ΔZ||/||Z|| {ratio:.4f} | gamma std {g.std().item():.3f} beta std {bb.std().item():.3f}')
            h = h_post
        else: h = h_pre
print('\ndiagnostics done')